In [1]:
# ============================================
# Startup Cell: Mount Drive + Prepare Data
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"
CONTENT_DIR = "/content"

SPLITS = ["train", "validation", "test"]
FEATURE_TYPES = ["gradient", "spatial", "frequency"]

# --------------------------------------------
# Copy feature CSVs from Drive to /content
# --------------------------------------------

print("Copying feature CSV files...")

for split in SPLITS:
    for feature_type in FEATURE_TYPES:
        filename = f"{split}_{feature_type}_features.csv"
        src = os.path.join(BASE_DRIVE_DIR, filename)
        dst = os.path.join(CONTENT_DIR, filename)

        if not os.path.exists(src):
            raise FileNotFoundError(f"Missing source file: {src}")

        shutil.copy(src, dst)
        print(f"Copied: {filename}")

print("\nFeature CSVs copied.")

# --------------------------------------------
# Verification
# --------------------------------------------

print("\nVerification:")
all_ok = True

for split in SPLITS:
    for feature_type in FEATURE_TYPES:
        filename = f"{split}_{feature_type}_features.csv"
        path = os.path.join(CONTENT_DIR, filename)
        exists = os.path.exists(path)
        print(f"{filename:<40} {'OK' if exists else 'MISSING'}")
        all_ok = all_ok and exists

print("\nAll files present." if all_ok else "\nSome files are missing.")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying feature CSV files...
Copied: train_gradient_features.csv
Copied: train_spatial_features.csv
Copied: train_frequency_features.csv
Copied: validation_gradient_features.csv
Copied: validation_spatial_features.csv
Copied: validation_frequency_features.csv
Copied: test_gradient_features.csv
Copied: test_spatial_features.csv
Copied: test_frequency_features.csv

Feature CSVs copied.

Verification:
train_gradient_features.csv              OK
train_spatial_features.csv               OK
train_frequency_features.csv             OK
validation_gradient_features.csv         OK
validation_spatial_features.csv          OK
validation_frequency_features.csv        OK
test_gradient_features.csv               OK
test_spatial_features.csv                OK
test_frequency_features.csv              OK

All files present.


In [2]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
#
# Purpose:
# Build final fixed-length feature vectors for one dataset split
# by combining gradient-based, spatial, and frequency-domain
# feature CSV files into a single classifier-ready table.
#
# Inputs:
#   /content/{SPLIT_NAME}_gradient_features.csv
#   /content/{SPLIT_NAME}_spatial_features.csv
#   /content/{SPLIT_NAME}_frequency_features.csv
#
# Assumptions:
# - Each input CSV contains one row per image for the selected split
# - Metadata columns are repeated in each CSV
# - Rows align across all three CSVs for the same images
# - Feature extraction has already been completed
#
# Output:
#   /content/{SPLIT_NAME}_feature_vectors.csv
#
# Final feature vector contents:
# - Metadata columns
# - 8 gradient-based features
# - 9 spatial features
# - 9 frequency-domain features
# - Total DIP features: 26
#
# Notes:
# - This notebook only builds feature vectors
# - No normalization is performed here
# - No classifier training is performed here
# ============================================



In [3]:
# ============================================
# Cell 1: Imports
# ============================================

import os
import pandas as pd
import numpy as np



In [4]:
# ============================================
# Cell 2: Define Input Paths
# ============================================

# -------------------------------------------------
# Select dataset split
# -------------------------------------------------
SPLIT_NAME = "train"   # options: "train", "validation", "test"

# -------------------------------------------------
# Define input paths
# -------------------------------------------------
GRADIENT_CSV  = f"/content/{SPLIT_NAME}_gradient_features.csv"
SPATIAL_CSV   = f"/content/{SPLIT_NAME}_spatial_features.csv"
FREQUENCY_CSV = f"/content/{SPLIT_NAME}_frequency_features.csv"

# -------------------------------------------------
# Define output path
# -------------------------------------------------
OUTPUT_CSV = f"/content/{SPLIT_NAME}_feature_vectors.csv"

# -------------------------------------------------
# Expected row counts
# -------------------------------------------------
EXPECTED_ROWS = {
    "train": 8400,
    "validation": 1800,
    "test": 1800
}

# -------------------------------------------------
# Display configuration
# -------------------------------------------------
print("SPLIT_NAME      =", SPLIT_NAME)
print("GRADIENT_CSV    =", GRADIENT_CSV)
print("SPATIAL_CSV     =", SPATIAL_CSV)
print("FREQUENCY_CSV   =", FREQUENCY_CSV)
print("OUTPUT_CSV      =", OUTPUT_CSV)
print("EXPECTED_ROWS   =", EXPECTED_ROWS[SPLIT_NAME])



SPLIT_NAME      = train
GRADIENT_CSV    = /content/train_gradient_features.csv
SPATIAL_CSV     = /content/train_spatial_features.csv
FREQUENCY_CSV   = /content/train_frequency_features.csv
OUTPUT_CSV      = /content/train_feature_vectors.csv
EXPECTED_ROWS   = 8400


In [5]:
# ============================================
# Cell 3: Load Feature CSVs
# ============================================

assert SPLIT_NAME in EXPECTED_ROWS, f"Invalid SPLIT_NAME: {SPLIT_NAME}"

for path in [GRADIENT_CSV, SPATIAL_CSV, FREQUENCY_CSV]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

df_grad = pd.read_csv(GRADIENT_CSV)
df_spat = pd.read_csv(SPATIAL_CSV)
df_freq = pd.read_csv(FREQUENCY_CSV)

print("Gradient shape: ", df_grad.shape)
print("Spatial shape:  ", df_spat.shape)
print("Frequency shape:", df_freq.shape)

expected_rows = EXPECTED_ROWS[SPLIT_NAME]
print("\nExpected rows:", expected_rows)



Gradient shape:  (8400, 13)
Spatial shape:   (8400, 14)
Frequency shape: (8400, 14)

Expected rows: 8400


In [6]:
# ============================================
# Cell 4: Validate Row Counts
# ============================================

if len(df_grad) != expected_rows:
    raise ValueError(f"Gradient CSV row count mismatch: expected {expected_rows}, got {len(df_grad)}")

if len(df_spat) != expected_rows:
    raise ValueError(f"Spatial CSV row count mismatch: expected {expected_rows}, got {len(df_spat)}")

if len(df_freq) != expected_rows:
    raise ValueError(f"Frequency CSV row count mismatch: expected {expected_rows}, got {len(df_freq)}")

print("PASS: all input CSVs have the expected row count")



PASS: all input CSVs have the expected row count


In [7]:
# ============================================
# Cell 5: Define Metadata and Feature Columns
# ============================================

metadata_cols = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

gradient_feature_cols = [
    "Mean Gradient",
    "Std Gradient",
    "Max Gradient",
    "Gradient Entropy",
    "Edge Density",
    "Orientation Mean",
    "Orientation Std",
    "Orientation Entropy"
]

spatial_feature_cols = [
    "Global Entropy",
    "Local Entropy Mean",
    "Local Entropy Std",
    "Intensity Mean",
    "Intensity Std",
    "Laplacian Variance",
    "Patch Variance Mean",
    "Patch Variance Std",
    "Noise Residual Energy"
]

frequency_feature_cols = [
    "Low Frequency Energy Ratio",
    "Mid Frequency Energy Ratio",
    "High Frequency Energy Ratio",
    "Radial Mean",
    "Radial Std",
    "Radial Entropy",
    "Spectral Centroid",
    "Spectral Bandwidth",
    "Log Spectrum Std"
]

print("Metadata columns: ", len(metadata_cols))
print("Gradient features:", len(gradient_feature_cols))
print("Spatial features: ", len(spatial_feature_cols))
print("Frequency features:", len(frequency_feature_cols))
print("Total DIP features:", len(gradient_feature_cols) + len(spatial_feature_cols) + len(frequency_feature_cols))



Metadata columns:  4
Gradient features: 8
Spatial features:  9
Frequency features: 9
Total DIP features: 26


In [8]:
# ============================================
# Cell 6: Verify Required Columns Exist
# ============================================

def check_columns(df, required_cols, df_name):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing columns: {missing}")
    print(f"{df_name}: all required columns present")

check_columns(df_grad, metadata_cols + gradient_feature_cols, "Gradient CSV")
check_columns(df_spat, metadata_cols + spatial_feature_cols, "Spatial CSV")
check_columns(df_freq, metadata_cols + frequency_feature_cols, "Frequency CSV")



Gradient CSV: all required columns present
Spatial CSV: all required columns present
Frequency CSV: all required columns present


In [9]:
# ============================================
# Cell 7: Verify Metadata Alignment Across CSVs
# ============================================

for col in metadata_cols:
    same_grad_spat = df_grad[col].equals(df_spat[col])
    same_grad_freq = df_grad[col].equals(df_freq[col])

    print(f"{col}:")
    print(f"  Gradient vs Spatial:   {same_grad_spat}")
    print(f"  Gradient vs Frequency: {same_grad_freq}")

    if not same_grad_spat or not same_grad_freq:
        raise ValueError(f"Mismatch detected in metadata column: {col}")

print("\nPASS: all metadata columns align across CSVs")



filename:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
class_label:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
source_dataset:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
subset:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True

PASS: all metadata columns align across CSVs


In [10]:
# ============================================
# Cell 8: Verify Split Labels Match SPLIT_NAME
# ============================================

unique_subsets_grad = sorted(df_grad["subset"].dropna().unique())
unique_subsets_spat = sorted(df_spat["subset"].dropna().unique())
unique_subsets_freq = sorted(df_freq["subset"].dropna().unique())

print("Gradient subset values: ", unique_subsets_grad)
print("Spatial subset values:  ", unique_subsets_spat)
print("Frequency subset values:", unique_subsets_freq)

expected_subset_value = SPLIT_NAME

if unique_subsets_grad != [expected_subset_value]:
    raise ValueError(f"Gradient CSV subset mismatch: expected [{expected_subset_value}], got {unique_subsets_grad}")

if unique_subsets_spat != [expected_subset_value]:
    raise ValueError(f"Spatial CSV subset mismatch: expected [{expected_subset_value}], got {unique_subsets_spat}")

if unique_subsets_freq != [expected_subset_value]:
    raise ValueError(f"Frequency CSV subset mismatch: expected [{expected_subset_value}], got {unique_subsets_freq}")

print("PASS: subset labels match SPLIT_NAME")



Gradient subset values:  ['train']
Spatial subset values:   ['train']
Frequency subset values: ['train']
PASS: subset labels match SPLIT_NAME


In [11]:
# ============================================
# Cell 9: Build Final Feature Vector DataFrame
# ============================================

df_feature_vectors = pd.concat(
    [
        df_grad[metadata_cols],
        df_grad[gradient_feature_cols],
        df_spat[spatial_feature_cols],
        df_freq[frequency_feature_cols]
    ],
    axis=1
)

print("Final feature vector shape:", df_feature_vectors.shape)
display(df_feature_vectors.head())



Final feature vector shape: (8400, 30)


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Noise Residual Energy,Low Frequency Energy Ratio,Mid Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,ai_diff_002852.png,ai,DiffusionDB,train,0.225056,0.382434,3.475054,3.217300,0.097748,-0.037221,...,0.000714,0.994641,0.004806,0.000552,5038657.0,66639156.0,0.049157,0.042543,0.554217,0.995231
1,rl_coco_001834.png,real,MS_COCO_2017,train,0.251470,0.391764,3.297258,3.473305,0.112564,0.248148,...,0.001410,0.988460,0.009412,0.002129,4558841.5,60337252.0,0.049157,0.035765,0.595008,0.932560
2,rl_coco_000745.png,real,MS_COCO_2017,train,0.176835,0.360221,3.563530,2.668197,0.079559,0.042095,...,0.001106,0.996467,0.002833,0.000700,14712400.0,197021008.0,0.049157,0.010006,0.303227,0.896367
3,rl_imgn_000309.png,real,ImageNet_1K_256,train,0.431584,0.599044,3.965402,4.068866,0.131790,-0.039664,...,0.003808,0.956267,0.035784,0.007949,2197359.5,28378992.0,0.098270,0.119204,1.281734,0.967731
4,rl_coco_000602.png,real,MS_COCO_2017,train,0.313866,0.333655,3.479813,3.953798,0.112183,0.090683,...,0.001584,0.991081,0.007367,0.001552,6841267.5,88922112.0,0.098270,0.046060,0.506594,0.931976


In [12]:
# ============================================
# Cell 10: Validate Final Feature Vector Table
# ============================================

expected_feature_count = (
    len(gradient_feature_cols) +
    len(spatial_feature_cols) +
    len(frequency_feature_cols)
)

actual_feature_count = len(df_feature_vectors.columns) - len(metadata_cols)

print("Expected feature count:", expected_feature_count)
print("Actual feature count:  ", actual_feature_count)

if actual_feature_count != expected_feature_count:
    raise ValueError("Feature count mismatch")

if len(df_feature_vectors) != expected_rows:
    raise ValueError(f"Final row count mismatch: expected {expected_rows}, got {len(df_feature_vectors)}")

print("\nClass counts:")
print(df_feature_vectors["class_label"].value_counts())

print("\nSource counts:")
print(df_feature_vectors["source_dataset"].value_counts())

print("\nSubset counts:")
print(df_feature_vectors["subset"].value_counts())

missing_counts = df_feature_vectors.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]

print("\nMissing values per column:")
if len(missing_counts) == 0:
    print("None")
else:
    print(missing_counts)

print("\nPASS: final feature vector table validated")



Expected feature count: 26
Actual feature count:   26

Class counts:
class_label
ai      4200
real    4200
Name: count, dtype: int64

Source counts:
source_dataset
DiffusionDB           2100
MS_COCO_2017          2100
ImageNet_1K_256       2100
SDXL_Generated_10K    2100
Name: count, dtype: int64

Subset counts:
subset
train    8400
Name: count, dtype: int64

Missing values per column:
None

PASS: final feature vector table validated


In [13]:
# ============================================
# Cell 11: Save Final Feature Vectors
# ============================================

df_feature_vectors.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
print("Shape:", df_feature_vectors.shape)



Saved: /content/train_feature_vectors.csv
Shape: (8400, 30)


In [14]:
# ============================================
# Cell 12: Quick Sanity Check
# ============================================

print("First 5 filenames:")
print(df_feature_vectors["filename"].head().to_list())

print("\nColumn names:")
for col in df_feature_vectors.columns:
    print(col)



First 5 filenames:
['ai_diff_002852.png', 'rl_coco_001834.png', 'rl_coco_000745.png', 'rl_imgn_000309.png', 'rl_coco_000602.png']

Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
Mid Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std
